# Technique 2

## CR3BP Case 0: halo-to-halo transfer (Earth-Moon rotating frame)

Three shooting strategies on the same 5000 random initial guesses: single shooting at the initial smoothing level $\epsilon_0=1$, stepwise homotopy $\epsilon=\epsilon_0,\epsilon_0/10,\ldots,10^{-5}$ with step halving on failure, and augmented shooting with the rational gate $q=W(R_s)$, $p=4$ and $R_0=0.1$, the configuration selected in Case 1. Convergence is judged by the physical shooting residual at the target smoothing level.

In [ ]:
import os
import sys
import platform

for name in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS',
             'VECLIB_MAXIMUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[name] = '1'

import numpy as np
import scipy
import heyoka as hy
from IPython.display import display, Markdown

hy.set_nthreads(1)

In [ ]:
# =========================
# Timing Settings
# =========================
from collections import Counter

cpu_samples = []
fast_cpus = []
if sys.platform == 'darwin':
    import ctypes
    import plistlib
    import subprocess
    tree = plistlib.loads(subprocess.check_output(
        ['/usr/sbin/ioreg', '-a', '-l', '-p', 'IODeviceTree', '-n', 'cpus', '-r']))
    pending = list(tree)
    while pending:
        node = pending.pop()
        if node.get('cluster-type') == b'P\x00' and 'cpu-id' in node:
            fast_cpus.append(int.from_bytes(node['cpu-id'], 'little'))
        pending.extend(node.get('IORegistryEntryChildren', []))
    lib = ctypes.CDLL('/usr/lib/libSystem.B.dylib')
    lib.pthread_set_qos_class_self_np.argtypes = [ctypes.c_uint, ctypes.c_int]
    lib.pthread_cpu_number_np.argtypes = [ctypes.POINTER(ctypes.c_size_t)]

def set_timing_priority():
    if sys.platform == 'darwin':
        assert lib.pthread_set_qos_class_self_np(0x21, 0) == 0

def sample_cpu():
    if sys.platform == 'darwin':
        cpu = ctypes.c_size_t()
        assert lib.pthread_cpu_number_np(ctypes.byref(cpu)) == 0
        cpu_samples.append(cpu.value)

records_by_method = {}
results_by_method = {}

In [ ]:
import pykep as pk

# =========================
# Setup: TOPS database problem P0 (cr3bp_hamilton_log)
# =========================
CASE = 'CR3BP_P0'
data = pk.trajopt.gym.tops_cr3bp_json['P0']
x0 = np.array(data['state_s'], dtype=float)
xf = np.array(data['state_f'], dtype=float)
MUnit = data['m_s']
LUnit = TUnit = 1.0          # the database states are already nondimensional
mu_nu = data['mu_cr3bp']
assert data['tof_bounds'][0] == data['tof_bounds'][1]   # fixed flight time

# =========================
# Nondimensionalize
# =========================
x0_nu = x0.tolist()
xf_nu = xf.tolist()
m0_nu = 1.0
t0_nu = 0.0
tf_nu = data['tof_bounds'][0] / TUnit
c1 = data['max_thrust'] * TUnit**2 / (MUnit * LUnit)   # thrust acceleration, nondimensional
c2 = data['veff'] * TUnit / LUnit                       # exhaust velocity, nondimensional
mass_kg = MUnit * data.get('MASS', 1.0)

print("Problem:", data['info'])
print(f"LUnit = {LUnit:.9e}, TUnit = {TUnit:.9e}, MUnit = {MUnit:.9e}")
print(f"mu_nu = {mu_nu:.16e}, c1 = {c1:.16e}, c2 = {c2:.16e}, tf_nu = {tf_nu:.16e}")
print("x0_nu =", np.asarray(x0_nu))
print("xf_nu =", np.asarray(xf_nu))


Problem: Halo 2 Halo in fixed tof
LUnit = 1.000000000e+00, TUnit = 1.000000000e+00, MUnit = 1.000000000e+00
mu_nu = 1.2150585609624041e-02, c1 = 3.0109995840114140e-01, c2 = 1.1564993721834320e+01, tf_nu = 5.0000000000000000e+00
x0_nu = [ 1.08099312  0.         -0.20235953  0.         -0.19895001  0.        ]
xf_nu = [ 1.16487809  0.         -0.11145304  0.         -0.20191923  0.        ]


In [ ]:
import cr3bp_hamilton_log

# State order: [x, y, z, vx, vy, vz, m, lx, ly, lz, lvx, lvy, lvz, lm]
# Parameter order: [mu, Tmax, exhaust velocity, epsilon, lambda0]; logarithmic smoothing of the throttle.
ta = cr3bp_hamilton_log.get_integrator('ta', tol=1.0e-12)

In [ ]:
# Numerical safeguards (as in the lunar notebook of Case 3): a propagation that does not reach the
# terminal time within integration_max_steps steps, a nonfinite state, or a homotopy variable below the
# admissible range return a large finite residual, so that MINPACK rejects the trial point and shrinks its
# trust region instead of evaluating pathological trajectories. Converged solutions use a few hundred steps.
integration_max_steps = 5000
BAD_RESIDUAL = 1.0e20

def _bad_residual(n):
    return np.full(n, BAD_RESIDUAL, dtype=float)

# =========================
# Define the shooting function
# =========================
def shooting(lam, eps = 1.0e-5, lambda0 = 1.0, out = False, Time_out = None):
    # lam[0:7] -> six initial state costates followed by the mass costate
    # function uses global variables ta, t0_nu, x0_nu, m0_nu, tf_nu, xf_nu, mu_nu, c1, c2
    ta.pars[:] = [mu_nu, c1, c2, eps, lambda0]
    ta.time = t0_nu
    ta.state[:7] = x0_nu + [m0_nu]
    ta.state[7:14] = lam[:]

    result = ta.propagate_until(tf_nu, max_steps=integration_max_steps)
    if ta.time != tf_nu or not np.all(np.isfinite(ta.state)):
        return _bad_residual(7)

    # Construct 7 equations: 6 terminal state errors + 1 terminal co-state lm = 0
    retval = [
        ta.state[0] - xf_nu[0],
        ta.state[1] - xf_nu[1],
        ta.state[2] - xf_nu[2],
        ta.state[3] - xf_nu[3],
        ta.state[4] - xf_nu[4],
        ta.state[5] - xf_nu[5],
        ta.state[13]
    ]

    if out:
        print("  Final Mass:", ta.state[6]*mass_kg)

    if Time_out is not None:
        test = Time_out

    return retval
Q_MIN_ALLOWED = -9.0          # eps(q) above 10 eps0 is meaningless; the lower side is clipped at eps_f in the dynamics (see below)

def shooting_augmented_gate(
    lam_augmented,
    eps0 = 1.0,             # initial smoothing level
    epsf = 1.0e-5,          # target smoothing level
    rho0 = 1.0e-2,          # gate threshold
    delta_rho = 1.0e-8,     # regularization for rho
    gate_p = 4.0,           # gate exponent
    gate_kind = "rational", # gate option
    lambda0 = 1.0,
    out = False,
    Time_out = None
):
    # lam_augmented[0:7] -> six initial state costates followed by the mass costate
    # lam_augmented[7]   -> homotopy variable q = (eps0 - eps) / (eps0 - epsf), i.e. eps is LINEAR in q:
    #   q = 0 -> eps = eps0
    #   q = 1 -> eps = epsf
    # (At fixed costates the shooting residual is linear in eps, so Newton's model is exact along q; with the
    #  logarithmic interpolation eps0^(1-q) epsf^q the exponential leaves a second-order remainder that dominates
    #  the small gate residual of the final phase and stalls the trust-region iteration.)

    q = lam_augmented[7]
    if not np.isfinite(q) or q < Q_MIN_ALLOWED:
        return _bad_residual(8)

    # The dynamics are integrated with eps never sharper than the target; the homotopy residual uses the raw q.
    eps = eps0 + min(q, 1.0) * (epsf - eps0)

    ta.pars[:] = [mu_nu, c1, c2, eps, lambda0]
    ta.time = t0_nu
    ta.state[:7] = x0_nu + [m0_nu]
    ta.state[7:14] = lam_augmented[:-1]
    result = ta.propagate_until(tf_nu, max_steps=integration_max_steps)
    if ta.time != tf_nu or not np.all(np.isfinite(ta.state)):
        return _bad_residual(8)

    # Terminal residual vector Phi
    Phi = np.array([
        ta.state[0] - xf_nu[0],
        ta.state[1] - xf_nu[1],
        ta.state[2] - xf_nu[2],
        ta.state[3] - xf_nu[3],
        ta.state[4] - xf_nu[4],
        ta.state[5] - xf_nu[5],
        ta.state[13]
    ], dtype=float)

    # Regularized RMS residual
    rho = np.sqrt(delta_rho**2 + np.mean(Phi * Phi))

    # Exponential and rational gate functions
    if gate_kind == "gaussian":
        w = np.exp(- (rho / rho0) ** gate_p)
    elif gate_kind == "rational":
        w = 1.0 / (1.0 + (rho / rho0) ** gate_p)
    else:
        raise ValueError(f"Unknown gate_kind: {gate_kind}")

    # Gate equation q = W(rho): q is the progress measured in eps, (eps0 - eps)/(eps0 - epsf)
    #   when rho is large -> w ~ 0 -> q ~ 0 -> eps ~ eps0
    #   when rho is small -> w ~ 1 -> q ~ 1 -> eps ~ epsf
    retval = list(Phi) + [q - w]

    if out:
        print(f"  Final Mass: {ta.state[6]*mass_kg}")
        print(f"  Final eps : {eps0 + q*(epsf - eps0):.16e}  (integrated with eps = {eps:.16e}, q = {q:.8f}, rho = {rho:.3e})")

    if Time_out is not None:
        test = Time_out

    return retval
# =========================
# Set parameters for minpack
# =========================
import time
from scipy.optimize import root
repeat_num_minpack  = 5000
# repeat_num_minpack  = 1

minpack_tol     = 1.0e-10
minpack_xtol    = 1.0e-10
minpack_maxfev  = 50000
minpack_factor  = 0.1
minpack_diag    = [1.0]*3 + [1.0]*3 + [1.0]*1
minpack_eps     = 0.
minpack_error   = 1.0e-7

np.set_printoptions(precision=16, suppress=True)

# =========================
# Convergence is judged by the physical residual at the target smoothing level only
# =========================
def physical_ok(lam, eps, lambda0 = 1.0):
    res = np.asarray(shooting(lam, eps, lambda0), dtype=float)
    return bool(np.all(np.isfinite(res)) and np.linalg.norm(res) < minpack_error
                and 0.0 < ta.state[6] <= 1.0)

In [ ]:

def prepare_timing():
    set_timing_priority()
    if sys.platform != 'darwin':
        return
    started = time.perf_counter()
    deadline = started + 2.0
    cpu = ctypes.c_size_t()
    while True:
        lib.pthread_cpu_number_np(ctypes.byref(cpu))
        if cpu.value in fast_cpus and time.perf_counter()-started >= 0.05:
            return
        if time.perf_counter() > deadline:
            raise RuntimeError('No Super-core scheduling during timing preparation.')

In [ ]:
# =========================
# One MINPACK solve, with optional patience
# =========================
# MINPACK hybr stops with status 4 or 5 when it measures insufficient progress over its last
# five Jacobian evaluations or ten iterations. In the sensitive three-body case this abort is
# premature for every method. With patience enabled, the solver is re-invoked from its
# returned point while it reports lack of progress, until two consecutive re-invocations bring
# no progress (no decrease of the residual norm by 10% and no advance of q) or the evaluation
# budget is used up. The same rule applies to all three methods of this notebook.
solver_patience = True
patience_budget = 5000
patience_stall  = 2

def solve_root(fun, x_init, args, diag, accept, jac=None, progress=None):
    # fun(x, *args) -> residual; jac(x, F0, *args) -> Jacobian at x given F0 = fun(x, *args) (optional: MINPACK then
    # runs hybrj instead of hybrd); progress(x) -> homotopy progress in [0, 1] (optional, for the patience rule).
    # Every residual evaluation is counted, including those inside jac, exactly as hybrd counts its own differences.
    x = np.asarray(x_init, dtype=float)
    count = dict(nfev=0, x=None, F=None)
    def fun_counted(x, *a):
        F = np.asarray(fun(x, *a), dtype=float)
        count['nfev'] += 1; count['x'] = np.array(x, copy=True); count['F'] = F
        return F
    def jac_counted(x, *a):
        F0 = count['F'] if (count['x'] is not None and np.array_equal(count['x'], x)) else fun_counted(x, *a)
        count['nfev'] += len(x)
        return jac(x, F0, *a)
    previous = None
    stall = 0
    while True:
        maxfev = max(patience_budget - count['nfev'], 50) if solver_patience else minpack_maxfev
        r = root(fun_counted, x, args=args, method='hybr', jac=(jac_counted if jac is not None else None), tol=minpack_tol,
                 options={'xtol': minpack_xtol, 'maxfev': maxfev, 'factor': minpack_factor,
                          'diag': diag, 'eps': minpack_eps})
        nfev = count['nfev']
        ok = bool(np.all(np.isfinite(r.x)) and accept(r.x))
        if ok or not solver_patience or r.status not in (4, 5) or nfev >= patience_budget \
                or not np.all(np.isfinite(r.x)):
            break
        fnorm = float(np.linalg.norm(r.fun)) if np.all(np.isfinite(r.fun)) else np.inf
        q_now = float(progress(r.x)) if progress is not None else 0.0
        if previous is not None:
            progressed = (q_now > previous[1] + 1.0e-3) or (fnorm < 0.9 * previous[0])
            stall = 0 if progressed else stall + 1
            if stall >= patience_stall:
                break
        previous = (fnorm, q_now)
        x = r.x
    return r, ok, nfev

In [ ]:
# =========================
# Common random initial guess pool (seed 322)
# =========================
rng = np.random.RandomState(322)
initial_guess_pool = np.empty((repeat_num_minpack, 7))
for i in range(repeat_num_minpack):
    initial_guess_pool[i] = (rng.random(7) - 0.5) * 2.0   # six state costates in (-1, 1)
    initial_guess_pool[i, 6] = rng.random()               # mass costate in (0, 1)
initial_guess_pool[:, 3:6] /= 300.0   # CR3BP velocity costates: (-1/300, 1/300), as in Technique 3 Case 4

# Augmented method: the same base guesses plus the initial homotopy variable (q0 = 0, i.e. eps = eps0)
augmented_initial_guess_pool = np.zeros((repeat_num_minpack, 8))
augmented_initial_guess_pool[:, :7] = initial_guess_pool
augmented_initial_guess_pool[:, 7] = 0.0

print(f"Generated {repeat_num_minpack} shared initial guesses.")


Generated 5000 shared initial guesses.


In [ ]:
# =========================
# Trial records
# =========================
# The trials of a method are independent. The saved outputs below were obtained by evaluating this
# same loop, one trial per call, in single-thread worker processes; the printed numbers are those of
# a serial run. The timing cells of each method are always measured serially.
def run_records(method, trial_fn):
    records = []
    for i in range(repeat_num_minpack):
        records.append(trial_fn(i))
        if (i + 1) % 500 == 0:
            print(f"{i+1} trials, {sum(r['accepted'] for r in records)} converged", flush=True)
    return records

def report_records(method, records):
    accepted = [r for r in records if r['accepted']]
    print(f"Solved: {len(accepted)} in {len(records)} ({len(accepted)/len(records):.4f})")
    if accepted:
        print(f"Average number of function calls for Minpack: {np.mean([r['nfev'] for r in records]):.0f}")
        print(f"Average number of function calls for convergent solution: {np.mean([r['nfev'] for r in accepted]):.0f}")
        best = max(accepted, key=lambda r: r['mass_kg'])
        print(f"Best final mass: {best['mass_kg']:.6f} kg (trial {best['trial']})")
    records_by_method[method] = records

## Method 1: normal shooting method (eps = 1)

In [ ]:
# =========================
# Method 1: single shooting at the initial smoothing level eps0
# =========================
epsf = 1.0
lambda0 = 1.0

def trial_single(i):
    x0 = initial_guess_pool[i].copy()
    r, ok, nfev = solve_root(shooting, x0, (epsf, lambda0, False, None), minpack_diag,
                             lambda x: physical_ok(x, epsf, lambda0))
    mass = float(ta.state[6] * mass_kg) if ok else None
    return dict(trial=int(i), accepted=ok, nfev=int(nfev), status=int(r.status),
                residual_norm=float(np.linalg.norm(r.fun)), eps=epsf, mass_kg=mass, x=r.x.tolist())

In [ ]:
records = run_records('Single_eps1', trial_single)
report_records('Single_eps1', records)

500 trials, 126 converged
1000 trials, 255 converged
1500 trials, 382 converged
2000 trials, 515 converged
2500 trials, 635 converged
3000 trials, 768 converged
3500 trials, 896 converged
4000 trials, 1016 converged
4500 trials, 1127 converged
5000 trials, 1249 converged
Solved: 1249 in 5000 (0.2498)
Average number of function calls for Minpack: 293
Average number of function calls for convergent solution: 325
Best final mass: 945.847413 kg (trial 4643)


In [ ]:
# =========================
# Computational time: all / successful / failed attempts
# =========================
timing_ids = np.arange(100)
accepted = np.array([r['accepted'] for r in records_by_method['Single_eps1']], dtype=bool)
successful_ids = timing_ids[accepted[timing_ids]]
failed_ids = timing_ids[~accepted[timing_ids]]
cpu_samples.clear()

In [ ]:
%%timeit -n 1 -r 3 -v timing_all prepare_timing()
for i in timing_ids:
    sample_cpu()
    trial_single(i)

17.4 s ± 16.6 ms per loop (mean ± std. dev. of 3 runs, 1 loop each)


In [ ]:
%%timeit -n 1 -r 3 -v timing_successful prepare_timing()
for i in successful_ids:
    sample_cpu()
    trial_single(i)

753 ms ± 5.16 ms per loop (mean ± std. dev. of 3 runs, 1 loop each)


In [ ]:
%%timeit -n 1 -r 3 -v timing_failed prepare_timing()
for i in failed_ids:
    sample_cpu()
    trial_single(i)

16.4 s ± 90.2 ms per loop (mean ± std. dev. of 3 runs, 1 loop each)


In [ ]:
all_ms = timing_all.average * 1000.0 / len(timing_ids)
p_success = accepted.mean()
results_by_method['Single_eps1'] = dict(
    method='Single shooting, epsilon=1', trials=len(accepted), accepted=int(accepted.sum()),
    first_ms=all_ms / p_success if p_success else float('inf'),
    successful_ms=timing_successful.average * 1000.0 / len(successful_ids) if len(successful_ids) else float('nan'),
    failed_ms=timing_failed.average * 1000.0 / len(failed_ids) if len(failed_ids) else float('nan'),
    all_ms=all_ms,
    nfev_successful=float(np.mean([r['nfev'] for r in records_by_method['Single_eps1'] if r['accepted']])) if accepted.any() else float('nan'),
    nfev_all=float(np.mean([r['nfev'] for r in records_by_method['Single_eps1']])),
    timing_trials=len(timing_ids), timing_successful_trials=len(successful_ids), timing_failed_trials=len(failed_ids),
    timeit_runs_seconds=timing_all.all_runs,
    timeit_successful_seconds=timing_successful.all_runs,
    timeit_failed_seconds=timing_failed.all_runs,
    cpu_samples=dict(Counter(cpu_samples)),
    fast_core_sample_pct=100.0 * sum(cpu in fast_cpus for cpu in cpu_samples) / len(cpu_samples) if cpu_samples else None)

## Method 2: consecutive iterations on the smoothing parameter

In [ ]:
# =========================
# Method 2: stepwise homotopy, eps = eps0, eps0/10, ..., 1e-5
# =========================
# Each stage solves the shooting problem at fixed eps from the previous solution. When a stage
# fails, the step in log10(eps) is halved and the stage is retried; after a successful stage the
# nominal step of one decade is restored. The path is abandoned below the minimum step.
eps0 = 1.0
epsf = 1.0e-5
lambda0 = 1.0
step_nominal = 1.0          # decades per stage
step_min = 1.0 / 32.0       # decades

def solve_for_eps(x_init, eps_val):
    return solve_root(shooting, x_init, (eps_val, lambda0, False, None), minpack_diag,
                      lambda x: physical_ok(x, eps_val, lambda0))

def trial_stepwise(i):
    x0 = initial_guess_pool[i].copy()
    r, ok, nfev = solve_for_eps(x0, eps0)
    logeps = np.log10(eps0)
    step = step_nominal
    if ok:
        x_guess = r.x
        while logeps > np.log10(epsf) + 1.0e-12:
            logeps_next = max(logeps - step, np.log10(epsf))
            r_next, ok_next, n_next = solve_for_eps(x_guess, 10.0**logeps_next)
            nfev += n_next
            if ok_next:
                logeps = logeps_next
                x_guess = r_next.x
                r = r_next
                step = step_nominal
            else:
                step *= 0.5
                if step < step_min:
                    break
    solved = bool(ok and logeps <= np.log10(epsf) + 1.0e-12)
    mass = None
    if solved:
        solved = physical_ok(r.x, epsf, lambda0)
        mass = float(ta.state[6] * mass_kg) if solved else None
    return dict(trial=int(i), accepted=solved, nfev=int(nfev), status=int(r.status),
                residual_norm=float(np.linalg.norm(r.fun)), eps=float(10.0**logeps), mass_kg=mass, x=r.x.tolist())

In [ ]:
records = run_records('Stepwise', trial_stepwise)
report_records('Stepwise', records)

500 trials, 67 converged
1000 trials, 137 converged
1500 trials, 197 converged
2000 trials, 264 converged
2500 trials, 338 converged
3000 trials, 415 converged
3500 trials, 477 converged
4000 trials, 540 converged
4500 trials, 607 converged
5000 trials, 670 converged
Solved: 670 in 5000 (0.1340)
Average number of function calls for Minpack: 1269
Average number of function calls for convergent solution: 3776
Best final mass: 973.327737 kg (trial 3518)


In [ ]:
# =========================
# Computational time: all / successful / failed attempts
# =========================
timing_ids = np.arange(100)
accepted = np.array([r['accepted'] for r in records_by_method['Stepwise']], dtype=bool)
successful_ids = timing_ids[accepted[timing_ids]]
failed_ids = timing_ids[~accepted[timing_ids]]
cpu_samples.clear()

In [ ]:
%%timeit -n 1 -r 3 -v timing_all prepare_timing()
for i in timing_ids:
    sample_cpu()
    trial_stepwise(i)

40.4 s ± 318 ms per loop (mean ± std. dev. of 3 runs, 1 loop each)


In [ ]:
%%timeit -n 1 -r 3 -v timing_successful prepare_timing()
for i in successful_ids:
    sample_cpu()
    trial_stepwise(i)

10 s ± 90.3 ms per loop (mean ± std. dev. of 3 runs, 1 loop each)


In [ ]:
%%timeit -n 1 -r 3 -v timing_failed prepare_timing()
for i in failed_ids:
    sample_cpu()
    trial_stepwise(i)

30.4 s ± 349 ms per loop (mean ± std. dev. of 3 runs, 1 loop each)


In [ ]:
all_ms = timing_all.average * 1000.0 / len(timing_ids)
p_success = accepted.mean()
results_by_method['Stepwise'] = dict(
    method='Stepwise', trials=len(accepted), accepted=int(accepted.sum()),
    first_ms=all_ms / p_success if p_success else float('inf'),
    successful_ms=timing_successful.average * 1000.0 / len(successful_ids) if len(successful_ids) else float('nan'),
    failed_ms=timing_failed.average * 1000.0 / len(failed_ids) if len(failed_ids) else float('nan'),
    all_ms=all_ms,
    nfev_successful=float(np.mean([r['nfev'] for r in records_by_method['Stepwise'] if r['accepted']])) if accepted.any() else float('nan'),
    nfev_all=float(np.mean([r['nfev'] for r in records_by_method['Stepwise']])),
    timing_trials=len(timing_ids), timing_successful_trials=len(successful_ids), timing_failed_trials=len(failed_ids),
    timeit_runs_seconds=timing_all.all_runs,
    timeit_successful_seconds=timing_successful.all_runs,
    timeit_failed_seconds=timing_failed.all_runs,
    cpu_samples=dict(Counter(cpu_samples)),
    fast_core_sample_pct=100.0 * sum(cpu in fast_cpus for cpu in cpu_samples) / len(cpu_samples) if cpu_samples else None)

## Method 3: augmented shooting method with rational gate (p = 4)

In [ ]:
# =========================
# Method 3: augmented shooting with the gate q = W(rho) (rational, p = 4); eps linear in q
# =========================
eps0 = 1.0
epsf = 1.0e-5

# parameters for the augmented residual (gate threshold R0 and exponent p, the same in every case)
gate_option = 'rational'
gate_p = 4.0
rho0 = 0.1
# Homotopy unknown: q = (eps0 - eps)/(eps0 - epsf), eps linear in q;
# Jacobian: MINPACK forward differences (hybrd)

# [other parameters]:
fd_eps = 0.
delta_rho = 1.0e-8
lambda0 = 1.0

def trial_gate(i):
    x0 = augmented_initial_guess_pool[i].copy()
    r, ok, nfev = solve_root(shooting_augmented_gate, x0,
                             (eps0, epsf, rho0, delta_rho, gate_p, gate_option, lambda0, False, None),
                             minpack_diag + [1.0], lambda x: physical_ok(x[:7], epsf, lambda0),
                             progress=lambda x: x[7])
    mass = float(ta.state[6] * mass_kg) if ok else None
    return dict(trial=int(i), accepted=ok, nfev=int(nfev), status=int(r.status),
                residual_norm=float(np.linalg.norm(r.fun)), eps=float(eps0 + r.x[7] * (epsf - eps0)),
                q=float(r.x[7]), mass_kg=mass, x=r.x.tolist())

In [ ]:
records = run_records('Gated_Rational_p4', trial_gate)
report_records('Gated_Rational_p4', records)

500 trials, 91 converged
1000 trials, 193 converged
1500 trials, 289 converged
2000 trials, 389 converged
2500 trials, 483 converged
3000 trials, 579 converged
3500 trials, 667 converged
4000 trials, 765 converged
4500 trials, 861 converged
5000 trials, 937 converged
Solved: 937 in 5000 (0.1874)
Average number of function calls for Minpack: 465
Average number of function calls for convergent solution: 694
Best final mass: 973.327737 kg (trial 3106)


In [ ]:
# =========================
# Computational time: all / successful / failed attempts
# =========================
timing_ids = np.arange(100)
accepted = np.array([r['accepted'] for r in records_by_method['Gated_Rational_p4']], dtype=bool)
successful_ids = timing_ids[accepted[timing_ids]]
failed_ids = timing_ids[~accepted[timing_ids]]
cpu_samples.clear()

In [ ]:
%%timeit -n 1 -r 3 -v timing_all prepare_timing()
for i in timing_ids:
    sample_cpu()
    trial_gate(i)

20 s ± 145 ms per loop (mean ± std. dev. of 3 runs, 1 loop each)


In [ ]:
%%timeit -n 1 -r 3 -v timing_successful prepare_timing()
for i in successful_ids:
    sample_cpu()
    trial_gate(i)

2.2 s ± 16.3 ms per loop (mean ± std. dev. of 3 runs, 1 loop each)


In [ ]:
%%timeit -n 1 -r 3 -v timing_failed prepare_timing()
for i in failed_ids:
    sample_cpu()
    trial_gate(i)

17.7 s ± 58.3 ms per loop (mean ± std. dev. of 3 runs, 1 loop each)


In [ ]:
all_ms = timing_all.average * 1000.0 / len(timing_ids)
p_success = accepted.mean()
results_by_method['Gated_Rational_p4'] = dict(
    method='Augmented, W_rat, p=4', trials=len(accepted), accepted=int(accepted.sum()),
    first_ms=all_ms / p_success if p_success else float('inf'),
    successful_ms=timing_successful.average * 1000.0 / len(successful_ids) if len(successful_ids) else float('nan'),
    failed_ms=timing_failed.average * 1000.0 / len(failed_ids) if len(failed_ids) else float('nan'),
    all_ms=all_ms,
    nfev_successful=float(np.mean([r['nfev'] for r in records_by_method['Gated_Rational_p4'] if r['accepted']])) if accepted.any() else float('nan'),
    nfev_all=float(np.mean([r['nfev'] for r in records_by_method['Gated_Rational_p4']])),
    timing_trials=len(timing_ids), timing_successful_trials=len(successful_ids), timing_failed_trials=len(failed_ids),
    timeit_runs_seconds=timing_all.all_runs,
    timeit_successful_seconds=timing_successful.all_runs,
    timeit_failed_seconds=timing_failed.all_runs,
    cpu_samples=dict(Counter(cpu_samples)),
    fast_core_sample_pct=100.0 * sum(cpu in fast_cpus for cpu in cpu_samples) / len(cpu_samples) if cpu_samples else None)

## Summary

In [ ]:

headings = ['Method', 'Success / 5000', 'Convergence [%]', 'Expected first [ms]',
            'Successful mean [ms]', 'Failed mean [ms]', 'All-trial mean [ms]', 'Nfev (all)', 'Nfev (converged)']
table = ['| '+' | '.join(headings)+' |', '| '+' | '.join(['---']*len(headings))+' |']
for value in results_by_method.values():
    values = [value['method'], value['accepted'], f"{100.*value['accepted']/value['trials']:.2f}"]
    values += [f"{value[k]:.3f}" for k in ('first_ms','successful_ms','failed_ms','all_ms')]
    values += [f"{value['nfev_all']:.1f}", f"{value['nfev_successful']:.1f}"]
    table.append('| '+' | '.join(map(str,values))+' |')
display(Markdown('\n'.join(table)))

| Method | Success / 5000 | Convergence [%] | Expected first [ms] | Successful mean [ms] | Failed mean [ms] | All-trial mean [ms] | Nfev (all) | Nfev (converged) |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Single shooting, epsilon=1 | 1249 | 24.98 | 696.440 | 30.131 | 218.435 | 173.971 | 293.1 | 324.5 |
| Stepwise | 670 | 13.40 | 3017.098 | 909.841 | 341.632 | 404.291 | 1269.1 | 3775.6 |
| Augmented, W_rat, p=4 | 937 | 18.74 | 1067.310 | 122.427 | 216.443 | 200.014 | 464.6 | 693.7 |